<a href="https://colab.research.google.com/github/Juansesalgado/Imputaci-n-de-datos/blob/main/Georreferenciaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx -q

import osmnx as ox
import geopandas as gpd

place = "Cartagena, Colombia"

tags = {"landuse": True}

# 👇 ESTA ES LA FUNCIÓN CORRECTA
gdf_suelo = ox.features_from_place(place, tags)

# Filtrar solo polígonos
gdf_suelo = gdf_suelo[gdf_suelo.geometry.type.isin(['Polygon','MultiPolygon'])]

print(gdf_suelo['landuse'].value_counts())

landuse
grass                      367
residential                153
industrial                 106
construction                47
brownfield                  32
retail                      30
recreation_ground           26
religious                   24
forest                      23
quarry                      20
farmland                    15
meadow                      12
commercial                  12
farmyard                    11
cemetery                    10
flowerbed                    9
garages                      7
proposed                     5
basin                        5
military                     4
village_green                4
greenfield                   4
harbour                      3
reservoir                    1
greenhouse_horticulture      1
aquaculture                  1
allotments                   1
Name: count, dtype: int64


In [ ]:

!pip install osmnx geopandas openpyxl -q


import pandas as pd
import geopandas as gpd
import osmnx as ox
from google.colab import files


archivo_excel = "base_convertida_itree.xlsx"
df = pd.read_excel(archivo_excel)


print("Columnas del archivo:")
print(df.columns)


df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

df = df.dropna(subset=["lat", "lon"]).copy()


df = df[(df["lat"].between(-90, 90)) & (df["lon"].between(-180, 180))].copy()


gdf_arboles = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

print("\nNúmero de árboles con coordenadas válidas:", len(gdf_arboles))


place = "Cartagena, Colombia"
tags = {"landuse": True}

gdf_suelo = ox.features_from_place(place, tags)

gdf_suelo = gdf_suelo[gdf_suelo.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

gdf_suelo = gdf_suelo[["landuse", "geometry"]].copy()

print("\nValores de landuse encontrados:")
print(gdf_suelo["landuse"].value_counts(dropna=False))


def clasificar_uso(valor):
    valor = str(valor).lower().strip()

    # Residencial
    if valor in ["residential", "garages"]:
        return "Residencial"

    # Industrial
    elif valor in ["industrial", "brownfield", "construction", "quarry"]:
        return "Industrial"

    # Comercial
    elif valor in ["commercial", "retail"]:
        return "Comercial"

    # Espacio verde
    elif valor in ["grass", "forest", "meadow", "village_green", "flowerbed", "recreation_ground"]:
        return "Espacio verde"

    # Agrícola
    elif valor in ["farmland", "farmyard", "greenhouse_horticulture", "allotments"]:
        return "Agricola"

    # Institucional
    elif valor in ["religious", "cemetery", "military"]:
        return "Institucional"

    # Infraestructura especial
    elif valor in ["basin", "reservoir", "harbour"]:
        return "Infraestructura"

    # Todo lo demás
    else:
        return "Otro"

gdf_suelo["Uso_iTree"] = gdf_suelo["landuse"].apply(clasificar_uso)

print("\nConteo de categorías reclasificadas:")
print(gdf_suelo["Uso_iTree"].value_counts(dropna=False))


if gdf_suelo.crs != gdf_arboles.crs:
    gdf_suelo = gdf_suelo.to_crs(gdf_arboles.crs)


gdf_final = gpd.sjoin(
    gdf_arboles,
    gdf_suelo[["geometry", "landuse", "Uso_iTree"]],
    how="left",
    predicate="within"
)

columnas_eliminar = [col for col in ["geometry", "index_right"] if col in gdf_final.columns]
resultado = pd.DataFrame(gdf_final.drop(columns=columnas_eliminar))

resultado["landuse"] = resultado["landuse"].fillna("sin_clasificar")
resultado["Uso_iTree"] = resultado["Uso_iTree"].fillna("Sin clasificar")


archivo_salida = "arboles_con_uso_suelo_itree.xlsx"
resultado.to_excel(archivo_salida, index=False, engine="openpyxl")

print(f"\n✅ Archivo exportado correctamente: {archivo_salida}")


files.download(archivo_salida)

Columnas del archivo:
Index(['Localidad', 'Biotipo', 'ID_Arbol', 'Altura', 'Shape_Length',
       'Shape_Area', 'Area_Ha', 'DBH_m', 'DBH_cm', 'crown_diameter', 'lat',
       'lon', 'Especie_Etiqueta', 'Altura_ft', 'DBH_in', 'Crown_diameter_ft'],
      dtype='object')

Número de árboles con coordenadas válidas: 186344

Valores de landuse encontrados:
landuse
grass                      367
residential                153
industrial                 106
construction                47
brownfield                  32
retail                      30
recreation_ground           26
religious                   24
forest                      23
quarry                      20
farmland                    15
meadow                      12
commercial                  12
farmyard                    11
cemetery                    10
flowerbed                    9
garages                      7
proposed                     5
basin                        5
military                     4
village_green       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(df.columns)

Index(['Localidad', 'Biotipo', 'ID_Arbol', 'Altura', 'Shape_Length',
       'Shape_Area', 'Area_Ha', 'DBH_m', 'DBH_cm', 'crown_diameter', 'lat',
       'lon', 'Especie_Etiqueta', 'Altura_ft', 'DBH_in', 'Crown_diameter_ft'],
      dtype='object')
